In [ ]:
!pip install opencv-python tqdm -q

In [ ]:
!pip install -q "Pillow==10.4.0" --force-reinstall
# import os; os.kill(os.getpid(), 9)   # restart runtime after install

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 32.7 MB/s eta 0:00:00


In [ ]:
import PIL
import torch
import torchvision
print("Pillow:", PIL.__version__)
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)

Pillow: 11.3.0
torch: 2.10.0+cu128
torchvision: 0.25.0+cu128


In [ ]:
!pip install -q torch torchvision
!git clone https://github.com/PeterL1n/RobustVideoMatting.git
import sys; sys.path.insert(0, '/content/RobustVideoMatting')

Cloning into 'RobustVideoMatting'...
remote: Enumerating objects: 211, done.
remote: Total 211 (delta 0), reused 0 (delta 0), pack-reused 211 (from 1)
Receiving objects: 100% (211/211), 9.00 MiB | 8.49 MiB/s, done.
Resolving deltas: 100% (81/81), done.


In [ ]:
VIDEO_PATH  = "/content/src.mp4"
OUTPUT_DIR  = "/content/output"

BATCH_SIZE  = 8      # frames processed at once — increase if GPU has more memory
                     # T4 (Colab free): 8-16 is safe
                     # A100 (Colab Pro): 32-64

BLUR_KSIZE  = 9      # edge softness (0 = hard edges)
RESIZE      = None   # e.g. (1920, 1080) or None to keep original
FMT         = "png"  # "png" or "jpg"
START_FRAME = 0
END_FRAME   = None   # None = all frames


#Imports & model

import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU")

# Load model
weights = DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
model   = deeplabv3_resnet50(weights=weights).to(DEVICE)
model.eval()

# Use half precision on GPU for ~2x speed boost
if DEVICE == "cuda":
    model = model.half()
    print("Using FP16 (half precision) for faster inference.")

print("Model ready.\n")

PERSON_CLASS = 15  # COCO person class

# ImageNet normalization constants as tensors (on device)
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
if DEVICE == "cuda":
    MEAN = MEAN.half()
    STD  = STD.half()

# Core batch segmentation

def frames_to_tensor(frames_bgr):
    """Convert a list of BGR numpy frames to a normalized GPU tensor (B, 3, H, W)."""
    tensors = []
    for f in frames_bgr:
        rgb = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        t   = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0  # (3, H, W)
        tensors.append(t)
    batch = torch.stack(tensors).to(DEVICE)   # (B, 3, H, W)
    if DEVICE == "cuda":
        batch = batch.half()
    batch = (batch - MEAN) / STD
    return batch


def segment_batch(frames_bgr, blur_ksize=9):
    """
    Segment a batch of BGR frames at once on GPU.

    Returns:
        fgr_list : list of (H, W, 3) uint8 — person in color, bg black
        pha_list : list of (H, W)    uint8 — white=person, black=bg
    """
    h, w = frames_bgr[0].shape[:2]

    with torch.no_grad():
        batch  = frames_to_tensor(frames_bgr)              # (B, 3, H, W)
        out    = model(batch)["out"]                       # (B, 21, H, W)
        preds  = out.argmax(dim=1).byte()                  # (B, H, W)
        masks  = (preds == PERSON_CLASS).float()           # (B, H, W) 0.0/1.0

    masks_np = (masks.cpu().numpy() * 255).astype(np.uint8)  # (B, H, W)

    fgr_list, pha_list = [], []
    for i, frame in enumerate(frames_bgr):
        binary = masks_np[i]

        # Soften edges
        if blur_ksize > 1:
            k   = blur_ksize if blur_ksize % 2 == 1 else blur_ksize + 1
            pha = cv2.GaussianBlur(binary, (k, k), 0)
        else:
            pha = binary

        mask_f = pha.astype(np.float32) / 255.0
        fgr    = np.clip(frame.astype(np.float32) * mask_f[..., None], 0, 255).astype(np.uint8)

        fgr_list.append(fgr)
        pha_list.append(pha)

    return fgr_list, pha_list



# Fast parallel save

def save_frame(args):
    """Save a single fgr + pha pair. Runs in a thread pool."""
    fgr, pha, fgr_path, pha_path, fmt = args
    png_params = [cv2.IMWRITE_PNG_COMPRESSION, 1]
    jpg_params = [cv2.IMWRITE_JPEG_QUALITY, 95]
    cv2.imwrite(fgr_path, fgr, png_params if fmt == "png" else jpg_params)
    cv2.imwrite(pha_path, pha)


# Main loop

def preprocess_video(video_path, output_dir, batch_size=8, blur_ksize=9,
                     resize=None, fmt="png", start_frame=0, end_frame=None):

    fgr_dir = Path(output_dir) / "fgr"
    pha_dir = Path(output_dir) / "pha"
    fgr_dir.mkdir(parents=True, exist_ok=True)
    pha_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS)
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    end_frame    = end_frame or total_frames

    print(f"Video      : {video_path}")
    print(f"Resolution : {width}x{height}  |  FPS: {fps:.2f}  |  Frames: {total_frames}")
    print(f"Processing : {start_frame} → {end_frame}  ({end_frame - start_frame} frames)")
    print(f"Batch size : {batch_size}  |  Device: {DEVICE}")
    print(f"Output     : {output_dir}")
    print()

    if start_frame > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    saved      = 0
    frame_buf  = []   # buffer of raw BGR frames for current batch

    # Thread pool for parallel disk writes (so GPU isn't waiting on I/O)
    executor = ThreadPoolExecutor(max_workers=4)
    futures  = []

    try:
        with tqdm(total=end_frame - start_frame, desc="Segmenting", unit="fr",
                  dynamic_ncols=True) as pbar:

            for global_idx in range(start_frame, end_frame):
                ret, frame = cap.read()
                if not ret:
                    break

                if resize:
                    frame = cv2.resize(frame, resize, interpolation=cv2.INTER_LANCZOS4)

                frame_buf.append(frame)

                # Process when buffer is full OR we're on the last frame
                if len(frame_buf) == batch_size or global_idx == end_frame - 1:
                    fgr_list, pha_list = segment_batch(frame_buf, blur_ksize)

                    # Submit saves to thread pool (non-blocking)
                    for fgr, pha in zip(fgr_list, pha_list):
                        fgr_path = str(fgr_dir / f"{saved:05d}.{fmt}")
                        pha_path = str(pha_dir  / f"{saved:05d}.png")
                        futures.append(
                            executor.submit(save_frame,
                                            (fgr, pha, fgr_path, pha_path, fmt))
                        )
                        saved += 1

                    pbar.update(len(frame_buf))
                    frame_buf = []

        # Wait for all saves to finish
        for f in futures:
            f.result()

    finally:
        cap.release()
        executor.shutdown(wait=True)

    print(f"\n Done!  Saved {saved} frames.")
    print(f"  fgr → {fgr_dir}")
    print(f"  pha → {pha_dir}")


# Run

preprocess_video(
    video_path  = VIDEO_PATH,
    output_dir  = OUTPUT_DIR,
    batch_size  = BATCH_SIZE,
    blur_ksize  = BLUR_KSIZE,
    resize      = RESIZE,
    fmt         = FMT,
    start_frame = START_FRAME,
    end_frame   = END_FRAME,
)

Device : cuda
Using FP16 (half precision) for faster inference.
Model ready.

Video      : /content/src.mp4
Resolution : 1920x1080  |  FPS: 59.99  |  Frames: 660
Processing : 0 → 660  (660 frames)
Batch size : 8  |  Device: cuda
Output     : /content/output



Segmenting: 100%|██████████| 660/660 [05:00<00:00,  2.20fr/s]



 Done!  Saved 660 frames.
  fgr → /content/output/fgr
  pha → /content/output/pha


In [ ]:
VIDEO_PATH  = "/content/src.mp4"
OUTPUT_DIR  = "/content/output"

BATCH_SIZE  = 8      # frames processed at once — increase if GPU has more memory
                     # T4 (Colab free): 8-16 is safe
                     # A100 (Colab Pro): 32-64

BLUR_KSIZE  = 9      # edge softness (0 = hard edges)
RESIZE      = None   # e.g. (1920, 1080) or None to keep original
FMT         = "png"  # "png" or "jpg"
START_FRAME = 0
END_FRAME   = None   # None = all frames

# Load RVM instead of DeepLabV3

import torch
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from model import MattingNetwork   # from the cloned repo

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")

# Load RVM with MobileNetV3 backbone (fastest, good quality)
# Alternative backbone: 'resnet50' — slower but higher quality
model = MattingNetwork('mobilenetv3').eval().to(DEVICE)
model.load_state_dict(
    torch.load('/content/rvm_mobilenetv3.pth', map_location=DEVICE)
)
if DEVICE == "cuda":
    model = model.half()
    print("Using FP16.")

print("RVM ready.\n")

# RVM carries its own recurrent state between frames — init once per video
rec = [None] * 4   # r1, r2, r3, r4  (hidden states for the GRU)


# New segment_batch using RVM


def frames_to_tensor_rvm(frames_bgr):
    """BGR list → normalized float tensor (B, 3, H, W) for RVM."""
    tensors = []
    for f in frames_bgr:
        rgb = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        t   = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0
        tensors.append(t)
    batch = torch.stack(tensors).to(DEVICE)
    if DEVICE == "cuda":
        batch = batch.half()
    return batch   # RVM normalizes internally — no manual mean/std needed


def segment_batch(frames_bgr, blur_ksize=9):
    """
    RVM-based matting. Replaces DeepLabV3 + guided filter entirely.
    rec[] carries temporal state so edges stay stable across frames.
    blur_ksize is kept for API compatibility but no longer used.
    """
    global rec

    with torch.no_grad():
        src = frames_to_tensor_rvm(frames_bgr)    # (B, 3, H, W)

        # RVM processes one frame at a time to thread the recurrent state
        fgr_tensors, pha_tensors = [], []
        for i in range(src.shape[0]):
            frame_t = src[i:i+1]                  # (1, 3, H, W)
            fgr_t, pha_t, *rec = model(frame_t, *rec, downsample_ratio=0.25)
            fgr_tensors.append(fgr_t)
            pha_tensors.append(pha_t)

        fgr_batch = torch.cat(fgr_tensors)        # (B, 3, H, W)  float [0,1]
        pha_batch = torch.cat(pha_tensors)        # (B, 1, H, W)  float [0,1]

    fgr_np  = (fgr_batch.float().cpu().numpy() * 255).astype(np.uint8)
    pha_np  = (pha_batch.float().cpu().numpy() * 255).astype(np.uint8)

    fgr_list, pha_list = [], []
    for i in range(len(frames_bgr)):
        # fgr: (3, H, W) → BGR (H, W, 3)
        fgr = cv2.cvtColor(fgr_np[i].transpose(1, 2, 0), cv2.COLOR_RGB2BGR)
        pha = pha_np[i, 0]                        # (H, W)
        fgr_list.append(fgr)
        pha_list.append(pha)

    return fgr_list, pha_list

# Fast parallel save

def save_frame(args):
    """Save a single fgr + pha pair. Runs in a thread pool."""
    fgr, pha, fgr_path, pha_path, fmt = args
    png_params = [cv2.IMWRITE_PNG_COMPRESSION, 1]
    jpg_params = [cv2.IMWRITE_JPEG_QUALITY, 95]
    cv2.imwrite(fgr_path, fgr, png_params if fmt == "png" else jpg_params)
    cv2.imwrite(pha_path, pha)


# Main loop

global rec; rec = [None] * 4

def preprocess_video(video_path, output_dir, batch_size=8, blur_ksize=9,
                     resize=None, fmt="png", start_frame=0, end_frame=None):

    fgr_dir = Path(output_dir) / "fgr"
    pha_dir = Path(output_dir) / "pha"
    fgr_dir.mkdir(parents=True, exist_ok=True)
    pha_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS)
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    end_frame    = end_frame or total_frames

    print(f"Video      : {video_path}")
    print(f"Resolution : {width}x{height}  |  FPS: {fps:.2f}  |  Frames: {total_frames}")
    print(f"Processing : {start_frame} → {end_frame}  ({end_frame - start_frame} frames)")
    print(f"Batch size : {batch_size}  |  Device: {DEVICE}")
    print(f"Output     : {output_dir}")
    print()

    if start_frame > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    saved      = 0
    frame_buf  = []   # buffer of raw BGR frames for current batch

    # Thread pool for parallel disk writes (so GPU isn't waiting on I/O)
    executor = ThreadPoolExecutor(max_workers=4)
    futures  = []

    try:
        with tqdm(total=end_frame - start_frame, desc="Segmenting", unit="fr",
                  dynamic_ncols=True) as pbar:

            for global_idx in range(start_frame, end_frame):
                ret, frame = cap.read()
                if not ret:
                    break

                if resize:
                    frame = cv2.resize(frame, resize, interpolation=cv2.INTER_LANCZOS4)

                frame_buf.append(frame)

                # Process when buffer is full OR we're on the last frame
                if len(frame_buf) == batch_size or global_idx == end_frame - 1:
                    fgr_list, pha_list = segment_batch(frame_buf, blur_ksize)

                    # Submit saves to thread pool (non-blocking)
                    for fgr, pha in zip(fgr_list, pha_list):
                        fgr_path = str(fgr_dir / f"{saved:05d}.{fmt}")
                        pha_path = str(pha_dir  / f"{saved:05d}.png")
                        futures.append(
                            executor.submit(save_frame,
                                            (fgr, pha, fgr_path, pha_path, fmt))
                        )
                        saved += 1

                    pbar.update(len(frame_buf))
                    frame_buf = []

        # Wait for all saves to finish
        for f in futures:
            f.result()

    finally:
        cap.release()
        executor.shutdown(wait=True)

    print(f"\n Done!  Saved {saved} frames.")
    print(f"  fgr → {fgr_dir}")
    print(f"  pha → {pha_dir}")


#  Run

preprocess_video(
    video_path  = VIDEO_PATH,
    output_dir  = OUTPUT_DIR,
    batch_size  = BATCH_SIZE,
    blur_ksize  = BLUR_KSIZE,
    resize      = RESIZE,
    fmt         = FMT,
    start_frame = START_FRAME,
    end_frame   = END_FRAME,
)

Device : cuda


FileNotFoundError: [Errno 2] No such file or directory: '/content/rvm_mobilenetv3.pth'

In [ ]:
!pip install -q diffusers transformers accelerate xformers

In [ ]:
import torch
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image
import cv2
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to(DEVICE)

#IMPORTANT for Colab (reduces VRAM)
pipe.enable_xformers_memory_efficient_attention()
pipe.enable_attention_slicing()

pipe.safety_checker = None

# Fixed seed → no flicker
generator = torch.Generator(device=DEVICE).manual_seed(42)

print("Diffusion ready (Colab optimized).")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
BG_PATH = "/content/canyon_0179.jpg"

bg = cv2.imread(BG_PATH)
if bg is None:
    raise RuntimeError("Background not found")

print("BG:", bg.shape)

In [ ]:
def composite(fgr, pha, bg):
    pha = pha.astype(np.float32) / 255.0
    pha = cv2.GaussianBlur(pha, (5,5), 0)
    pha = np.expand_dims(pha, axis=2)

    bg_resized = cv2.resize(bg, (fgr.shape[1], fgr.shape[0]))

    return (fgr * pha + bg_resized * (1 - pha)).astype(np.uint8)

In [ ]:
def diffusion_refine(frame):
    small = cv2.resize(frame, (640, 360))

    pil = Image.fromarray(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))

    out = pipe(
        prompt="",
        image=pil,
        strength=0.2,
        guidance_scale=1.0,
        generator=generator
    ).images[0]

    out = cv2.cvtColor(np.array(out), cv2.COLOR_RGB2BGR)

    return cv2.resize(out, (frame.shape[1], frame.shape[0]))

In [ ]:
from pathlib import Path
from tqdm import tqdm
import gc

fgr_dir = Path("/content/output/fgr")
pha_dir = Path("/content/output/pha")
out_dir = Path("/content/output/final_frames")
out_dir.mkdir(parents=True, exist_ok=True)

fgr_files = sorted(fgr_dir.glob("*.png"))

print(f"Total frames: {len(fgr_files)}")

for i, fgr_path in enumerate(tqdm(fgr_files)):
    pha_path = pha_dir / fgr_path.name

    fgr = cv2.imread(str(fgr_path))
    pha = cv2.imread(str(pha_path), 0)

    # Step 1: Composite
    comp = composite(fgr, pha, bg)

    # Step 2: Diffusion every 2 frames (speed + stability)
    if i % 2 == 0:
        final = diffusion_refine(comp)
    else:
        final = comp

    cv2.imwrite(str(out_dir / f"{i:05d}.png"), final)

    # Prevent Colab crashes
    if i % 20 == 0:
        torch.cuda.empty_cache()
        gc.collect()

print("Done.")

In [ ]:
!ffmpeg -y -framerate 30 -i /content/output/final_frames/%05d.png \
-c:v libx264 -pix_fmt yuv420p /content/final_output.mp4

In [ ]:
# =========================================================
# 1. INSTALL
# =========================================================
!pip install -q diffusers transformers accelerate xformers

# =========================================================
# 2. IMPORTS
# =========================================================
import torch
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
from PIL import Image

from diffusers import StableDiffusionImg2ImgPipeline

# =========================================================
# 3. SETTINGS
# =========================================================
FGR_DIR = "/content/output/fgr"
PHA_DIR = "/content/output/pha"
BG_PATH = "/content/canyon_0179.jpg"
OUT_DIR = "/content/output/final_frames"

# =========================================================
# 4. LOAD DIFFUSION
# =========================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to(DEVICE)

pipe.enable_xformers_memory_efficient_attention()
pipe.enable_attention_slicing()
pipe.safety_checker = None

generator = torch.Generator(device=DEVICE).manual_seed(42)

print("Diffusion ready")

# =========================================================
# 5. LOAD & ENHANCE BACKGROUND (ONLY ONCE)
# =========================================================
bg = cv2.imread(BG_PATH)
if bg is None:
    raise RuntimeError("Background not found")

# resize for diffusion (faster)
bg_small = cv2.resize(bg, (768, 432))

pil_bg = Image.fromarray(cv2.cvtColor(bg_small, cv2.COLOR_BGR2RGB))

styled_bg = pipe(
    prompt="",
    negative_prompt="",
    image=pil_bg,
    strength=0.4,          # can increase (0.3–0.6)
    guidance_scale=1.0,
    generator=generator
).images[0]

styled_bg = cv2.cvtColor(np.array(styled_bg), cv2.COLOR_RGB2BGR)

print("Background enhanced once")

# =========================================================
# 6. COMPOSITE FUNCTION
# =========================================================
def composite(fgr, pha, bg):
    pha = pha.astype(np.float32) / 255.0
    pha = cv2.GaussianBlur(pha, (5,5), 0)
    pha = np.expand_dims(pha, axis=2)

    bg_resized = cv2.resize(bg, (fgr.shape[1], fgr.shape[0]))

    return (fgr * pha + bg_resized * (1 - pha)).astype(np.uint8)

# =========================================================
# 7. PROCESS ALL FRAMES (VERY FAST)
# =========================================================
fgr_dir = Path(FGR_DIR)
pha_dir = Path(PHA_DIR)
out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

fgr_files = sorted(fgr_dir.glob("*.png"))

print(f"🎬 Total frames: {len(fgr_files)}")

for i, fgr_path in enumerate(tqdm(fgr_files)):
    pha_path = pha_dir / fgr_path.name

    fgr = cv2.imread(str(fgr_path))
    pha = cv2.imread(str(pha_path), 0)

    final = composite(fgr, pha, styled_bg)

    cv2.imwrite(str(out_dir / f"{i:05d}.png"), final)

print("Frames done")

# =========================================================
# 8. VIDEO
# =========================================================
!ffmpeg -y -framerate 30 -i /content/output/final_frames/%05d.png \
-c:v libx264 -pix_fmt yuv420p /content/final_output.mp4

print("Done saved to /content/final_output.mp4")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Diffusion ready


  0%|          | 0/20 [00:00<?, ?it/s]

Background enhanced once
🎬 Total frames: 660


100%|██████████| 660/660 [03:07<00:00,  3.51it/s]


Frames done
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-l